In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.43 GB)

Index cache: /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


# 2. Load Corpora and Build/Load Indices

In [3]:
import pandas as pd
from tqdm.notebook import tqdm
import pickle
import re
from rank_bm25 import BM25Okapi
import bm25s
import Stemmer

class BM25Index:
    """BM25 index for keyword search over legal documents.

    Supports Swiss federal laws (SR) and court decisions (BGE).
    """

    def __init__(
        self,
        documents: list[dict] | None = None,
        text_field: str = "text",
        citation_field: str = "citation",
    ):
        """Initialize BM25 index.

        Args:
            documents: List of document dictionaries
            text_field: Key for document text in dict
            citation_field: Key for citation string in dict
        """
        self.text_field = text_field
        self.citation_field = citation_field

        self.documents: list[dict] = []
        self.index = bm25s.BM25()
        self.stemmer = Stemmer.Stemmer("german")
        self._tokenized_corpus: list[list[str]] = []

        if documents:
            self.build(documents)

    def build(self, documents: list[dict]) -> None:
        """Build BM25 index from documents.

        Args:
            documents: List of document dictionaries
        """
        self.documents = documents

        corpus_texts = [doc.get(self.text_field, "") for doc in self.documents]
        corpus_tokens = bm25s.tokenize(
                            corpus_texts,
                            stopwords='de',
                            stemmer=self.stemmer,
                        )

        # Tokenize all documentsf

        # Build BM25 index
        self.index.index(corpus_tokens)

    def search(
        self,
        query: str,
        top_k: int = 10,
        return_scores: bool = False,
    ) -> list[dict]:
        """Search the index with a query.

        Args:
            query: Search query string
            top_k: Number of results to return
            return_scores: Whether to include BM25 scores in results

        Returns:
            List of matching documents (with optional scores)
        """
        if self.index is None:
            raise ValueError("Index not built. Call build() first.")

        queries = [query]
        
        query_tokens = bm25s.tokenize(
            queries,
            stopwords='de',
            stemmer=self.stemmer,
        )
        
        if not query_tokens:
            return []

        # retrieve 返回索引，再映射回原始 corpus
        results, scores = self.index.retrieve(
            query_tokens,
            k=top_k,
        )
        
        output = []
        for q_idx, query in enumerate(queries):
            hits = []
            for rank in range(results.shape[1]):
                doc_idx = results[q_idx, rank]
                hits.append({
                    "rank": rank + 1,
                    "score": round(float(scores[q_idx, rank]), 4),
                    "text": self.documents[doc_idx]["text"],
                    "citation": self.documents[doc_idx]["citation"],
                })
            output.append({"query": query, "hits": hits})
        return output

    def save(self, path: Path | str) -> None:
        """Save index to disk.

        Args:
            path: Path to save index (creates .pkl file)
        """
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        self.index.save(path)

        doc_path = str(path) + '.doc'

        data = {}
        data['text_field'] = self.text_field
        data['citation_field'] = self.citation_field
        data["documents"] = self.documents
        
        with open(doc_path, "wb") as f:
            pickle.dump(data, f)

    @classmethod
    def load(cls, path: Path | str) -> "BM25Index":
        """Load index from disk.

        Args:
            path: Path to saved index

        Returns:
            Loaded BM25Index instance
        """
        path = Path(path)

        doc_path = str(path) + ".doc"
        with open(doc_path, "rb") as f:
            data = pickle.load(f)

        instance = cls(
            text_field=data["text_field"],
            citation_field=data.get("citation_field", "citation"),
        )
        instance.documents = data["documents"]
        instance.index = bm25s.BM25.load(str(path))

        return instance


def load_csv_corpus(
    csv_path: Path,
    chunk_size: int = 100_000,
    max_rows: int | None = None
) -> list[dict]:
    """Load CSV corpus into list of dicts with progress bar.
    
    Args:
        csv_path: Path to CSV file with 'citation' and 'text' columns
        chunk_size: Rows to process per chunk (for memory efficiency)
        max_rows: Optional limit on rows (for testing with smaller corpus)
    
    Returns:
        List of {"citation": str, "text": str} dicts
    """
    documents = []
    
    # Count rows for progress bar (fast line count)
    print(f"Counting rows in {csv_path.name}...")
    with open(csv_path, encoding='utf-8') as f:
        total_rows = sum(1 for _ in f) - 1  # minus header
    
    if max_rows:
        total_rows = min(total_rows, max_rows)
    print(f"Total rows to load: {total_rows:,}")
    
    rows_loaded = 0
    with tqdm(total=total_rows, desc=f"Loading {csv_path.name}") as pbar:
        for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
            for _, row in chunk.iterrows():
                if max_rows and rows_loaded >= max_rows:
                    break
                documents.append({
                    "citation": str(row["citation"]),
                    "text": str(row["text"]) if pd.notna(row["text"]) else ""
                })
                rows_loaded += 1
            pbar.update(min(len(chunk), total_rows - pbar.n))
            if max_rows and rows_loaded >= max_rows:
                break
    
    return documents


def get_or_build_index(
    name: str,
    csv_path: Path,
    index_path: Path,
    force_rebuild: bool = False,
    max_rows: int | None = None
) -> BM25Index:
    """Load cached index or build from CSV.
    
    Args:
        name: Index name for logging
        csv_path: Path to corpus CSV
        index_path: Path to cache index pickle
        force_rebuild: If True, rebuild even if cache exists
        max_rows: Optional row limit (for testing with smaller corpus)
    
    Returns:
        BM25Index instance
    """
    # Use cached index if available and not forcing rebuild
    if index_path.exists() and not force_rebuild:
        print(f"Loading cached {name} index from {index_path}")
        index = BM25Index.load(index_path)
        print(f"  Loaded {len(index.documents):,} documents")
        return index
    
    # Check CSV exists
    if not csv_path.exists():
        print(f"Warning: {csv_path} not found. Creating empty index.")
        return BM25Index(documents=[])
    
    # Load corpus from CSV
    print(f"\n{'='*50}")
    print(f"Building {name} index from {csv_path}")
    print(f"{'='*50}")
    documents = load_csv_corpus(csv_path, max_rows=max_rows)
    
    if not documents:
        print(f"Warning: No documents loaded. Creating empty index.")
        return BM25Index(documents=[])
    
    # Build BM25 index
    print(f"\nBuilding BM25 index for {len(documents):,} documents...")
    index = BM25Index(
        documents=documents,
        text_field="text",
        citation_field="citation"
    )
    print(f"Index built successfully!")
    
    # Cache index for future runs
    if not KAGGLE_ENV:
        print(f"Saving index to {index_path}...")
        index.save(index_path)
        print(f"Index cached.")
    
    return index

In [4]:
# Load or build laws index
# Laws CSV: ~45MB, ~269K rows
# Build time: ~30 seconds | Load from cache: <1 second

laws_index = get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Uncomment to test with smaller corpus
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(test_results)

Loading cached laws index from /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/laws_index.pkl
  Loaded 175,933 documents

Laws index: 175,933 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Vertrag': 1 results
[{'query': 'Vertrag', 'hits': [{'rank': 1, 'score': 3.1854, 'text': '6 Ist das BAKOM Registerbetreiberin, so untersteht der Vertrag dem öffentlichen Recht (verwaltungsrechtlicher Vertrag); ist die Aufgabe an einen Dritten übertragen, so untersteht der Vertrag dem Privatrecht (privatrechtlicher Vertrag).', 'citation': 'Art. 17 Abs. 6 VID'}, {'rank': 2, 'score': 3.124, 'text': 'Durch diesen Vertrag werden die entgegenstehenden Bestimmungen früherer Verträge aufgehoben.', 'citation': 'Art. 16 Abs. 1 414.110.1'}, {'rank': 3, 'score': 3.124, 'text': '1 Durch Vertrag kann die Verpflichtung zum Abschluss eines künftigen Vertrages begründet werden.', 'citation': 'Art. 22 Abs. 1 OR'}]}]


In [5]:
test_results = laws_index.search(laws_index.documents[0]['text'], top_k=3)
if test_results:
    print(test_results)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

[{'query': 'Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäude des Bundesrathauses im roten Quartier der Stadt Bern, mit Nr. 229 bezeichnet, nebst den in demselben enthaltenen Einrichtungen und Mobilien, welche der Einwohnergemeinde angehören, und unter Vorbehalt der im Artikel 62 von der Einwohnergemeinde reservierten Einrichtungen und Gegenstände;\nb. den zwischen den Seitenflügeln des Bundesrathauses und nördlich von dem Mittelbau desselben befindlichen innern Hof von ungefähr 25 000 Quadratfuss Oberfläche.\n Derselbe wird abgetreten bis zu einer in Verlängerung der Nordfassaden der Seitenflügel gezogenen Linie.\n Der in diesem Hofe befindliche Brunnen verbleibt der Einwohnergemeinde, welche denselben in gutem Zustande erhalten und ohne Genehmigung des Bundesrates an dem jetzigen baulichen Zustand mit Inbegriff der Statuen keine Veränderung vornehmen soll.\n Sie wird den Brunnen wie bis anhin mit Wasser versehen.\n Die

In [6]:
# Load or build courts index
# Courts CSV: ~2.3GB, ~2.5M rows
# Full corpus build time: ~15-20 minutes | Load from cache: ~10 seconds
# Full corpus can have peak memory during build: ~8-16GB

courts_index = get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV,
    index_path=COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Change to use bigger corpus
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")

Loading cached courts index from /root/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/courts_index.pkl
  Loaded 2,476,315 documents

Courts index: 2,476,315 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Meinungsfreiheit': 1 results
  Top result: N/A


In [7]:
def extract_citations_from_text(text: str) -> list[str]:
    """Extract citations from any text (tool output or final answer)."""
    citations = []
    
    # SR pattern: SR followed by number (optionally with article)
    sr_matches = re.findall(
        r"SR\s*\d{3}(?:\.\d+)?(?:\s+Art\.?\s*\d+[a-z]?)?",
        text,
        re.IGNORECASE
    )
    citations.extend(sr_matches)
    
    # BGE pattern: BGE volume section page
    bge_matches = re.findall(
        r"BGE\s+\d{1,3}\s+[IVX]+[a-z]?\s+\d+(?:\s+E\.\s*\d+[a-z]?)?",
        text,
        re.IGNORECASE
    )
    citations.extend(bge_matches)
    
    # Art. pattern: Art. X LAW (e.g., Art. 1 ZGB, Art. 41 OR)
    art_matches = re.findall(
        r"Art\.?\s+\d+[a-z]?\s+(?:Abs\.?\s*\d+\s+)?[A-Z]{2,}",
        text,
        re.IGNORECASE
    )
    citations.extend(art_matches)
    
    return list(set(citations))

In [8]:
test_df = pd.read_csv('../data/test_rewrite_001.csv')
id_l = []
citation_l = []

for id, q in zip(test_df['query_id'].tolist(), test_df['query'].tolist()):
    print("query len:", len(q))
    id_l.append(id)
    citations = []

    courts_query = []
    test_results = courts_index.search(q, top_k=10)
    if test_results:
        for hit in test_results[0]['hits']:
            citations.append(hit['citation'])
            courts_query.append(hit['text'])
            for c in extract_citations_from_text(hit['text']):
                citations.append(c)


    test_results = laws_index.search(q, top_k=15)
    if test_results:
        for hit in test_results[0]['hits']:
            citations.append(hit['citation'])

    citations = list(set(citations))
    citation_l.append(';'.join(citations))
    print(id)

result_df = pd.DataFrame({'query_id':id_l, 'predicted_citations':citation_l})
result_df.to_csv("../data/result.csv", index=False)

query len: 394


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

test_040


In [9]:
# test_df = pd.read_csv(QUERY_FILE)

# id_l = []
# citation_l = []

# for id, q in zip(test_df['query_id'].tolist(), test_df['query'].tolist()):
#     print("query len:", len(q))
#     id_l.append(id)
#     citations = []
#     test_results = laws_index.search(q, top_k=20)
#     if test_results:
#         for hit in test_results[0]['hits']:
#             citations.append(hit['citation'])

#     test_results = courts_index.search(q, top_k=20)
#     if test_results:
#         for hit in test_results[0]['hits']:
#             citations.append(hit['citation'])

#     citations = list(set(citations))
#     citation_l.append(';'.join(citations))
#     print(id)

# result_df = pd.DataFrame({'query_id':id_l, 'predicted_citations':citation_l})
# result_df.to_csv("../data/result.csv")

In [10]:


TRAIN_HITS_LAW_PKL = "../data/processed/search_result_law_hits.pkl"
TRAIN_HITS_COURTS_PKL = "../data/processed/search_result_courts_hits.pkl"
def get_or_build_train_hits():
    law_hits_l = []
    courts_hits_l = []
    if os.path.exists(TRAIN_HITS_LAW_PKL) and os.path.exists(TRAIN_HITS_COURTS_PKL):
        with open(TRAIN_HITS_LAW_PKL, "rb") as inf:
            law_hits_l = pickle.load(inf)

        with open(TRAIN_HITS_COURTS_PKL, "rb") as inf:
            courts_hits_l = pickle.load(inf)
    else:
        id_l = []
        train_df = pd.read_csv("../data/train.csv")
    
        for id, q, gold_citations in tqdm(zip(train_df['query_id'].tolist(), 
                                              train_df['query'].tolist(), 
                                              train_df['gold_citations'].tolist()), total=len(train_df)):
            # print("query len:", len(q))
            id_l.append(id)
            citations = []
            law_hits = []
            test_results = laws_index.search(q, top_k=1000)
            if test_results:
                law_hits_l.append(test_results[0]['hits'])
            else:
                law_hits_l.append([])
        
            test_results = courts_index.search(q, top_k=1000)
            if test_results: 
                courts_hits_l.append(test_results[0]['hits'])
            else:
                courts_hits_l.append([])

            with open(TRAIN_HITS_LAW_PKL, "wb+") as of:
                pickle.dump(law_hits_l, of)

            with open(TRAIN_HITS_COURTS_PKL, "wb+") as of:
                pickle.dump(courts_hits_l, of)

    print(len(law_hits_l), len(law_hits_l[0]))
    return law_hits_l, courts_hits_l

In [11]:
def cal_recall(law_hists_l, courts_hits_l, gold_citations_l):
    recalls = []
    for law_hits, courts_hits,gold_citations in zip(law_hists_l, courts_hits_l, gold_citations_l):
        
        all_citation = []
        for hit in law_hits:
            all_citation.append(hit['citation'])
        for hit in courts_hits:
            all_citation.append(hit['citation'])

        hits = len(set(all_citation) & set(gold_citations))

        print("gold_citations.len:", len(gold_citations), ", hits.len:", hits)
        recall = hits / len(gold_citations)
        recalls.append(recall)
        
    mean_recall = np.mean(recalls)
    return mean_recall

In [ ]:
law_hits_l, courts_hits_l = get_or_build_train_hits()

train_df = pd.read_csv("../data/train.csv")

print(cal_recall(law_hits_l, courts_hits_l, train_df['gold_citations'].apply(lambda x: x.split(";"))))


  0%|          | 0/1139 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]